In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Insert example cohort rows into the Lakebase control table (one row per source database x table).
# src_type picks the source adapter (sqlserver | oracle | postgresql | mysql); load_type picks the load
# strategy (ct | full | cursor; cdc, and cursor on non-SQL Server sources, are placeholders that fail per
# table with a clear message).
# Source reads use Spark JDBC with the per-database credential secret named in `secret_key`
# (JSON {"user", "password"}) - no UC connection. src_port NULL = the engine's default port.
# select_list must list explicit columns (`*` is not supported).
#
# load_type 'ct' (SQL Server only) prereqs per table: CT enabled, PK present,
# CHANGE_RETENTION > expected seed time (snapshot isolation is not required).
# load_type 'cursor' (SQL Server) follows a monotonically increasing cursor_column (rowversion recommended;
# identity / decimal / date-time types also work) and needs a PK. Deletes are not captured.
# load_type 'full' reloads the whole table every run (data-only atomic replace); no PK needed. Oracle and
# MySQL need their JDBC driver on the job cluster (see resources/lakefed_ingest.yml).
cols = ["job_name", "task_collection", "src_type", "src_host", "src_port", "secret_key",
        "src_database", "src_schema", "src_table", "sink_catalog", "sink_schema", "sink_table",
        "primary_key", "load_type", "select_list", "use_remote_query", "ct_enabled",
        "tier", "priority", "task_enabled",
        # Partitioned seed / full load (large tables): size-bounded range partitions on a numeric/date/
        # datetime column. Leave load_partitioned False for small tables.
        "load_partitioned", "partition_col", "partition_size_mb",
        # Optional extra Spark JDBC read options (JSON object), e.g. '{"fetchsize": "50000"}'.
        "jdbc_options",
        # load_type 'cursor': the high-watermark column.
        "cursor_column",
        # Optional consolidated sink: rows with client_id set share one sink table (MERGE on client_id + PK,
        # seeds replace only that client's rows). NULL = the row owns its sink.
        "client_id"]
rows = [
    # SQL Server Change Tracking, partitioned seed
    ("lakefed_ingest", "example_cohort", "sqlserver", "sqlserver-01.example.com", 1433,
     "tenant-db-001", "tenant_db_001", "dbo", "orders", "main", "bronze_tenant_db_001", "orders",
     "order_id", "ct", "order_id, customer_id, status, updated_at", False, True, "gold", 1, True,
     True, "order_id", 512, None, None, None),
    ("lakefed_ingest", "example_cohort", "sqlserver", "sqlserver-02.example.com", 1433,
     "tenant-db-002", "tenant_db_002", "dbo", "orders", "main", "bronze_tenant_db_002", "orders",
     "order_id", "ct", "order_id, customer_id, status, updated_at", False, True, "gold", 1, True,
     False, None, None, None, None, None),
    # SQL Server watermark increments on a rowversion column (no Change Tracking on this table)
    ("lakefed_ingest", "example_cohort", "sqlserver", "sqlserver-01.example.com", 1433,
     "tenant-db-001", "tenant_db_001", "dbo", "invoices", "main", "bronze_tenant_db_001", "invoices",
     "invoice_id", "cursor", "invoice_id, customer_id, amount, row_ver", False, False, "gold", 2, True,
     False, None, None, None, "row_ver", None),
    # Consolidated sink: two client databases' dbo.leases into ONE table main.bronze.leases (by client_id)
    ("lakefed_ingest", "example_cohort", "sqlserver", "sqlserver-01.example.com", 1433,
     "tenant-db-001", "tenant_db_001", "dbo", "leases", "main", "bronze", "leases",
     "lease_id", "ct", "lease_id, unit_id, rent, updated_at", False, True, "gold", 1, True,
     False, None, None, None, None, "tenant_db_001"),
    ("lakefed_ingest", "example_cohort", "sqlserver", "sqlserver-02.example.com", 1433,
     "tenant-db-002", "tenant_db_002", "dbo", "leases", "main", "bronze", "leases",
     "lease_id", "ct", "lease_id, unit_id, rent, updated_at", False, True, "gold", 1, True,
     False, None, None, None, None, "tenant_db_002"),
    # Full reload of a small SQL Server reference table (no PK needed)
    ("lakefed_ingest", "example_cohort", "sqlserver", "sqlserver-01.example.com", 1433,
     "tenant-db-001", "tenant_db_001", "dbo", "country_codes", "main", "bronze_tenant_db_001",
     "country_codes", None, "full", "code, name", False, False, "silver", 5, True,
     False, None, None, None, None, None),
    # Full reload from PostgreSQL (default port 5432), with a larger fetch size
    ("lakefed_ingest", "example_cohort", "postgresql", "postgres-01.example.com", None,
     "app-db-001", "app", "public", "products", "main", "bronze_app", "products",
     None, "full", "product_id, sku, price", False, False, "silver", 5, True,
     False, None, None, '{"fetchsize": "50000"}', None, None),
    # Partitioned full reload from Oracle (service name in src_database; upper-case stored names)
    ("lakefed_ingest", "example_cohort", "oracle", "oracle-01.example.com", None,
     "erp-db-001", "ERPPDB", "ERP", "GL_LINES", "main", "bronze_erp", "gl_lines",
     None, "full", "LINE_ID, ACCOUNT, AMOUNT, POSTED_AT", False, False, "silver", 5, True,
     True, "LINE_ID", 512, None, None, None),
]
placeholders = ", ".join(["%s"] * len(cols))
with store.connection.cursor() as cur:
    cur.executemany(
        f"insert into lakefed_ingest.control ({', '.join(cols)}) values ({placeholders})", rows)
print(f"inserted {len(rows)} control rows")